# BBBC019 cell-region segmentation with U-Net

This portfolio notebook downloads the official **BBBC019v2** wound-healing microscopy dataset, prepares a reproducible source-disjoint split, trains an assignment-style U-Net, and exports a complete evaluation report.

The official BBBC019 page catalogs 171 manually segmented DIC images from eight source datasets. The completed run used 165 verified image-mask pairs: 120 images from four sources for training, 19 images from two unseen sources for validation, and 26 images from two other unseen sources for held-out testing. The loader records the discovered counts in the split because the downloaded Init data may expose 22 pairs rather than the 28 cataloged images. Dataset files are downloaded directly from the Broad Bioimage Benchmark Collection and are not added to Git.

## 1. Runtime and paths

Choose a GPU runtime in Colab. PyTorch, NumPy, Pillow, and Matplotlib are normally preinstalled. This cell checks the runtime and creates paths under `/content`.

In [ ]:
from pathlib import Path
import importlib.util

required = ('torch', 'numpy', 'PIL', 'matplotlib')
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f"Missing packages: {missing}. Run: %pip install torch numpy Pillow matplotlib")

DATA_DIR = Path('/content/data/BBBC019')
ARCHIVE_DIR = Path('/content/data/BBBC019_archives')
SPLIT_PATH = Path('/content/splits/bbbc019.json')
RUN_DIR = Path('/content/runs/bbbc019')
RESULTS_DIR = Path('/content/results/bbbc019')
print('Paths configured')

## 2. Download and validate BBBC019v2

The eight archives come from the [official BBBC019 page](https://bbbc.broadinstitute.org/BBBC019). Downloads total roughly a few hundred MB. The code safely extracts them, pairs each microscopy image with its `_manual.png` mask, checks dimensions, validates source counts, and saves the exact discovered counts.

In [ ]:
"""Download, validate, and split the official BBBC019v2 dataset."""

import argparse
import json
import random
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile

from PIL import Image


BASE_URL = "https://data.broadinstitute.org/bbbc/BBBC019"
ARCHIVES = {
    "TScratch": "TScratch.zip",
    "Melanoma": "Melanoma.zip",
    "Init": "Init.zip",
    "SN15": "SN15.zip",
    "Scatter": "Scatter.zip",
    "Microfluidics": "Microfluidic.zip",
    "HEK293": "HEK293.zip",
    "MDCK": "MDCK.zip",
}
EXPECTED_COUNTS = {
    "TScratch": 24,
    "Melanoma": 20,
    "Init": 28,
    "SN15": 54,
    "Scatter": 6,
    "Microfluidics": 13,
    "HEK293": 12,
    "MDCK": 14,
}
ALLOWED_PAIRED_COUNTS = {
    source: ({22, 28} if source == "Init" else {count})
    for source, count in EXPECTED_COUNTS.items()
}
SOURCE_ALIASES = {"Microfluidic": "Microfluidics"}
DEFAULT_VAL_SOURCES = ("Microfluidics", "Scatter")
DEFAULT_TEST_SOURCES = ("HEK293", "MDCK")
IMAGE_EXTENSIONS = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}


def _safe_extract(archive_path, destination):
    destination = Path(destination).resolve()
    with ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe path in {archive_path}: {member.filename}")
        archive.extractall(destination)


def download_bbbc019(data_dir, archive_dir=None):
    """Download and extract the eight official BBBC019v2 archives."""
    data_dir = Path(data_dir)
    archive_dir = Path(archive_dir or data_dir.parent / "BBBC019_archives")
    data_dir.mkdir(parents=True, exist_ok=True)
    archive_dir.mkdir(parents=True, exist_ok=True)
    for source, filename in ARCHIVES.items():
        archive_path = archive_dir / filename
        if not archive_path.exists():
            url = f"{BASE_URL}/{filename}"
            print(f"Downloading {source}: {url}")
            urllib.request.urlretrieve(url, archive_path)
        existing = list(data_dir.rglob(f"{source}/images"))
        if source == "Microfluidics":
            existing += list(data_dir.rglob("Microfluidic/images"))
        if not existing:
            print(f"Extracting {archive_path.name}")
            _safe_extract(archive_path, data_dir)
    pairs = discover_bbbc019_pairs(data_dir)
    counts = Counter(pair["source"] for pair in pairs)
    unexpected = {
        source: counts.get(source, 0)
        for source, allowed in ALLOWED_PAIRED_COUNTS.items()
        if counts.get(source, 0) not in allowed
    }
    extra_sources = sorted(set(counts) - set(ALLOWED_PAIRED_COUNTS))
    if unexpected or extra_sources:
        raise ValueError(
            "Unexpected BBBC019 pair counts. "
            f"Allowed {ALLOWED_PAIRED_COUNTS}; found {dict(counts)}; "
            f"extra sources {extra_sources}"
        )
    print(f"BBBC019 ready: {len(pairs)} paired images in {data_dir}")
    return pairs


def discover_bbbc019_pairs(data_dir):
    """Find official image/manual-mask pairs after archive extraction."""
    data_dir = Path(data_dir)
    if not data_dir.is_dir():
        raise FileNotFoundError(f"Missing BBBC019 directory: {data_dir}")
    pairs = []
    for image_dir in sorted(data_dir.rglob("images")):
        if "__MACOSX" in image_dir.parts:
            continue
        manual_dir = image_dir.parent / "manual"
        if not manual_dir.is_dir():
            continue
        source = SOURCE_ALIASES.get(image_dir.parent.name, image_dir.parent.name)
        for scan in sorted(image_dir.iterdir()):
            if not scan.is_file() or scan.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            mask = manual_dir / f"{scan.stem}_manual.png"
            if not mask.is_file():
                raise ValueError(f"Missing manual mask for {scan}")
            with Image.open(scan) as image, Image.open(mask) as label:
                if image.size != label.size:
                    raise ValueError(f"Image and mask sizes differ: {scan}, {mask}")
                image.verify()
                label.verify()
            pairs.append({
                "scan": scan.relative_to(data_dir).as_posix(),
                "label": mask.relative_to(data_dir).as_posix(),
                "source": source,
            })
    if not pairs:
        raise ValueError(f"No BBBC019 image/manual pairs found below {data_dir}")
    names = [(pair["source"], Path(pair["scan"]).stem.casefold()) for pair in pairs]
    duplicates = [name for name, count in Counter(names).items() if count > 1]
    if duplicates:
        raise ValueError(f"Duplicate BBBC019 image identifiers: {duplicates[:5]}")
    return pairs


def make_source_stratified_split(pairs, train_fraction=0.70, val_fraction=0.15, seed=42):
    """Split each of the eight source datasets so all remain represented."""
    if train_fraction <= 0 or val_fraction <= 0 or train_fraction + val_fraction >= 1:
        raise ValueError("train and validation fractions must be positive and sum to less than 1")
    by_source = defaultdict(list)
    for pair in pairs:
        if "source" not in pair:
            raise ValueError("Every BBBC019 pair must contain a source")
        by_source[pair["source"]].append(pair)
    split = {"train": [], "val": [], "test": []}
    for source in sorted(by_source):
        group = sorted(by_source[source], key=lambda pair: pair["scan"])
        random.Random(f"{seed}:{source}").shuffle(group)
        if len(group) < 3:
            raise ValueError(f"Source {source} needs at least three images")
        train_count = min(len(group) - 2, max(1, round(len(group) * train_fraction)))
        val_count = min(len(group) - train_count - 1, max(1, round(len(group) * val_fraction)))
        split["train"].extend(group[:train_count])
        split["val"].extend(group[train_count:train_count + val_count])
        split["test"].extend(group[train_count + val_count:])
    for name in split:
        split[name].sort(key=lambda pair: (pair["source"], pair["scan"]))
    split.update({
        "dataset": "BBBC019v2",
        "seed": seed,
        "train_fraction": train_fraction,
        "val_fraction": val_fraction,
        "source_counts": {
            name: dict(sorted(Counter(pair["source"] for pair in split[name]).items()))
            for name in ("train", "val", "test")
        },
    })
    return split


def make_source_holdout_split(
    pairs, val_sources=DEFAULT_VAL_SOURCES, test_sources=DEFAULT_TEST_SOURCES, seed=42
):
    """Create source-disjoint splits for an honest cross-domain evaluation."""
    val_sources, test_sources = set(val_sources), set(test_sources)
    if not val_sources or not test_sources or val_sources & test_sources:
        raise ValueError("Validation and test sources must be nonempty and disjoint")
    available = {pair.get("source") for pair in pairs}
    requested = val_sources | test_sources
    if None in available or not requested <= available:
        raise ValueError(f"Requested sources {sorted(requested)} not found in {sorted(available)}")
    train_sources = available - requested
    if not train_sources:
        raise ValueError("At least one source must remain for training")
    split = {"train": [], "val": [], "test": []}
    for pair in pairs:
        source = pair["source"]
        name = "val" if source in val_sources else "test" if source in test_sources else "train"
        split[name].append(pair)
    for name in split:
        split[name].sort(key=lambda pair: (pair["source"], pair["scan"]))
    split.update({
        "dataset": "BBBC019v2",
        "split_strategy": "source_disjoint",
        "seed": seed,
        "train_sources": sorted(train_sources),
        "val_sources": sorted(val_sources),
        "test_sources": sorted(test_sources),
        "source_counts": {
            name: dict(sorted(Counter(pair["source"] for pair in split[name]).items()))
            for name in ("train", "val", "test")
        },
    })
    return split

In [ ]:
pairs = download_bbbc019(DATA_DIR, ARCHIVE_DIR)
split = make_source_holdout_split(pairs, seed=42)
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
SPLIT_PATH.write_text(json.dumps(split, indent=2) + '\n', encoding='utf-8')
print({name: len(split[name]) for name in ('train', 'val', 'test')})
print(json.dumps(split['source_counts'], indent=2))

## 3. Preprocessing and augmentation

Images are converted to grayscale, padded to square without changing aspect ratio, resized to 572 x 572, and normalized to `[0, 1]`. Masks use nearest-neighbor resizing. Training uses paired horizontal/vertical flips, 90-degree rotations, zoom, and image-only gamma correction. Validation and test samples are deterministic.

In [ ]:
"""Validate paired cell images, create a reproducible split, and load samples."""

import argparse
import json
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

try:
    from torch.utils.data import Dataset
except ImportError:  # Pair validation can run before PyTorch is installed.
    Dataset = object


IMAGE_EXTENSIONS = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}


def _files_by_stem(directory):
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing directory: {directory}")
    files = {}
    for path in directory.iterdir():
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            key = path.stem.casefold()
            if key in files:
                raise ValueError(f"Duplicate image stem in {directory}: {path.stem}")
            files[key] = path
    return files


def discover_pairs(data_dir):
    data_dir = Path(data_dir)
    scans = _files_by_stem(data_dir / "scans")
    labels = _files_by_stem(data_dir / "labels")
    if not scans:
        raise ValueError(f"No scan images found in {data_dir / 'scans'}")
    if scans.keys() != labels.keys():
        missing_labels = sorted(scans.keys() - labels.keys())
        missing_scans = sorted(labels.keys() - scans.keys())
        raise ValueError(
            f"Unpaired files: missing labels for {missing_labels}; "
            f"missing scans for {missing_scans}"
        )
    pairs = []
    for key in sorted(scans):
        scan, label = scans[key], labels[key]
        with Image.open(scan) as image, Image.open(label) as mask:
            if image.size != mask.size:
                raise ValueError(f"Image and mask sizes differ: {scan.name}, {label.name}")
            image.verify()
            mask.verify()
        pairs.append({"scan": scan.name, "label": label.name})
    return pairs


def make_split(pairs, train_fraction=0.8, val_fraction=0.1, seed=42):
    if train_fraction <= 0 or val_fraction <= 0 or train_fraction + val_fraction >= 1:
        raise ValueError("train and validation fractions must be positive and sum to less than 1")
    if len(pairs) < 3:
        raise ValueError("At least three image/mask pairs are needed")
    shuffled = list(pairs)
    random.Random(seed).shuffle(shuffled)
    train_count = min(len(shuffled) - 2, max(1, round(len(shuffled) * train_fraction)))
    val_count = min(len(shuffled) - train_count - 1, max(1, round(len(shuffled) * val_fraction)))
    return {
        "seed": seed,
        "train_fraction": train_fraction,
        "val_fraction": val_fraction,
        "train": sorted(shuffled[:train_count], key=lambda pair: pair["scan"]),
        "val": sorted(shuffled[train_count:train_count + val_count], key=lambda pair: pair["scan"]),
        "test": sorted(shuffled[train_count + val_count:], key=lambda pair: pair["scan"]),
    }


def load_split(path, data_dir):
    with open(path, encoding="utf-8") as handle:
        split = json.load(handle)
    if split.get("dataset") == "BBBC019v2":
        from bbbc019 import discover_bbbc019_pairs
        discovered = discover_bbbc019_pairs(data_dir)
    else:
        discovered = discover_pairs(data_dir)
    current = {(pair["scan"], pair["label"]) for pair in discovered}
    train = {(pair["scan"], pair["label"]) for pair in split["train"]}
    val = {(pair["scan"], pair["label"]) for pair in split["val"]}
    test = {(pair["scan"], pair["label"]) for pair in split["test"]}
    if not train or not val or not test or train & val or train & test or val & test or train | val | test != current:
        raise ValueError("Split must contain every pair exactly once, with nonempty train, val, and test sets")
    if any(len(group) != len(split[name]) for name, group in (("train", train), ("val", val), ("test", test))):
        raise ValueError("Split contains duplicate pairs")
    return split


def _resolve_pair_path(data_dir, value, default_directory):
    """Resolve new relative paths and legacy filename-only split entries."""
    relative = Path(value)
    direct = data_dir / relative
    if direct.is_file():
        return direct
    legacy = data_dir / default_directory / relative
    if legacy.is_file():
        return legacy
    raise FileNotFoundError(f"Could not resolve {value} below {data_dir}")


def _pad_pair_to_square(image, mask):
    """Preserve aspect ratio by padding before square model resizing."""
    side = max(image.size)
    if image.size == (side, side):
        return image, mask
    left = (side - image.width) // 2
    top = (side - image.height) // 2
    fill = int(np.median(np.asarray(image)))
    padded_image = Image.new("L", (side, side), color=fill)
    padded_mask = Image.new("L", (side, side), color=0)
    padded_image.paste(image, (left, top))
    padded_mask.paste(mask, (left, top))
    return padded_image, padded_mask


class Cell_data(Dataset):
    """PyTorch DataLoader compatible dataset; masks contain class IDs 0 or 1."""

    def __init__(self, data_dir, pairs, size=572, augment=False):
        if size < 320:
            raise ValueError("size must be at least 320 to produce a 128x128 or larger mask")
        self.data_dir = Path(data_dir)
        self.pairs = list(pairs)
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        pair = self.pairs[index]
        scan_path = _resolve_pair_path(self.data_dir, pair["scan"], "scans")
        label_path = _resolve_pair_path(self.data_dir, pair["label"], "labels")
        with Image.open(scan_path) as source:
            image = source.convert("L")
        with Image.open(label_path) as source:
            mask = source.convert("L")
        image, mask = _pad_pair_to_square(image, mask)
        image = image.resize((self.size, self.size), Image.Resampling.BILINEAR)
        mask = mask.resize((self.size, self.size), Image.Resampling.NEAREST)

        if self.augment:
            if random.random() < 0.5:
                image, mask = ImageOps.mirror(image), ImageOps.mirror(mask)
            if random.random() < 0.5:
                image, mask = ImageOps.flip(image), ImageOps.flip(mask)
            turns = random.randrange(4)
            if turns:
                angle = 90 * turns
                image, mask = image.rotate(angle), mask.rotate(angle)

            # Zoom in with the same crop on both images; masks stay categorical.
            if random.random() < 0.5:
                crop_size = round(self.size / random.uniform(1.05, 1.25))
                left = random.randrange(self.size - crop_size + 1)
                top = random.randrange(self.size - crop_size + 1)
                box = (left, top, left + crop_size, top + crop_size)
                image = image.crop(box).resize((self.size, self.size), Image.Resampling.BILINEAR)
                mask = mask.crop(box).resize((self.size, self.size), Image.Resampling.NEAREST)

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        if self.augment and random.random() < 0.5:
            image_array = np.power(image_array, random.uniform(0.7, 1.4)).astype(np.float32)
        image_array = image_array[None, :, :]
        mask_array = (np.asarray(mask) > 0).astype(np.int64)
        return image_array, mask_array


CellDataset = Cell_data  # Backward-compatible name for existing scripts.

## 4. Assignment-style U-Net

The model starts with 64 channels and uses unpadded 3 x 3 convolutions, max pooling, transpose convolutions, and center-cropped skip connections. A 572 x 572 image produces a 388 x 388 mask. Raw logits are optimized with cross-entropy loss.

In [ ]:
"""Assignment-style U-Net with valid 3x3 convolutions and cropped skip connections."""

import torch
from torch import nn


def center_crop(tensor, height, width):
    """Crop the spatial center of an image, mask, or feature map."""
    source_height, source_width = tensor.shape[-2:]
    if height > source_height or width > source_width:
        raise ValueError(f"Cannot crop {source_height}x{source_width} to {height}x{width}")
    top = (source_height - height) // 2
    left = (source_width - width) // 2
    return tensor[..., top:top + height, left:left + width]


class twoConvBlock(nn.Module):
    """Valid convolution, ReLU, valid convolution, batch norm, ReLU."""

    def __init__(self, input_channel, output_channel):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(input_channel, output_channel, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.Conv2d(output_channel, output_channel, kernel_size=3),
            nn.BatchNorm2d(output_channel),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)


class downStep(nn.Module):
    """Contracting path: four conv/pool stages and one bottleneck block."""

    def __init__(self, base_channels=64):
        super().__init__()
        c = base_channels
        self.blocks = nn.ModuleList(
            [twoConvBlock(1, c), twoConvBlock(c, 2*c),
             twoConvBlock(2*c, 4*c), twoConvBlock(4*c, 8*c)]
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = twoConvBlock(8*c, 16*c)

    def forward(self, x):
        skips = []
        for block in self.blocks:
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        return self.bottleneck(x), skips


class upStep(nn.Module):
    """Transpose convolutions followed by cropped skip concatenation."""

    def __init__(self, base_channels=64):
        super().__init__()
        c = base_channels
        self.upsamples = nn.ModuleList(
            [nn.ConvTranspose2d(16*c, 8*c, 2, stride=2),
             nn.ConvTranspose2d(8*c, 4*c, 2, stride=2),
             nn.ConvTranspose2d(4*c, 2*c, 2, stride=2),
             nn.ConvTranspose2d(2*c, c, 2, stride=2)]
        )
        self.blocks = nn.ModuleList(
            [twoConvBlock(16*c, 8*c), twoConvBlock(8*c, 4*c),
             twoConvBlock(4*c, 2*c), twoConvBlock(2*c, c)]
        )

    def forward(self, x, skips):
        for upsample, block, skip in zip(self.upsamples, self.blocks, reversed(skips)):
            x = upsample(x)
            skip = center_crop(skip, *x.shape[-2:])
            x = block(torch.cat((skip, x), dim=1))
        return x


class UNet(nn.Module):
    def __init__(self, base_channels=64):
        super().__init__()
        if base_channels < 1:
            raise ValueError("base_channels must be positive")
        self.down = downStep(base_channels)
        self.up = upStep(base_channels)
        self.classifier = nn.Conv2d(base_channels, 2, kernel_size=1)

    def forward(self, x):
        x, skips = self.down(x)
        return self.classifier(self.up(x, skips))  # Raw logits for CrossEntropyLoss.

In [ ]:
"""Aggregate segmentation metrics across an entire data loader."""

import torch
from torch import nn
import torch.nn.functional as F



def _metrics_from_counts(loss_sum, pixels, tp, fp, fn, tn, samples):
    return {
        "loss": loss_sum / pixels,
        "pixel_accuracy": (tp + tn) / pixels,
        "dice": (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else 1.0,
        "iou": (tp / (tp + fp + fn)) if (tp + fp + fn) else 1.0,
        "precision": (tp / (tp + fp)) if (tp + fp) else 1.0,
        "recall": (tp / (tp + fn)) if (tp + fn) else 1.0,
        "samples": samples,
    }


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    loss_sum = pixels = tp = fp = fn = tn = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        labels = center_crop(labels, *logits.shape[-2:])
        loss_sum += criterion(logits, labels).item()
        predictions = logits.argmax(dim=1)
        tp += ((predictions == 1) & (labels == 1)).sum().item()
        fp += ((predictions == 1) & (labels == 0)).sum().item()
        fn += ((predictions == 0) & (labels == 1)).sum().item()
        tn += ((predictions == 0) & (labels == 0)).sum().item()
        pixels += labels.numel()
    if not pixels:
        raise ValueError("Cannot evaluate an empty dataset")
    return _metrics_from_counts(loss_sum, pixels, tp, fp, fn, tn, len(loader.dataset))


@torch.no_grad()
def evaluate_detailed(model, loader, device):
    """Return aggregate, per-source, and per-image semantic metrics."""
    model.eval()
    totals = {"loss": 0.0, "pixels": 0, "tp": 0, "fp": 0, "fn": 0, "tn": 0, "samples": 0}
    source_totals = {}
    per_image = []
    offset = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        labels = center_crop(labels, *logits.shape[-2:])
        predictions = logits.argmax(dim=1)
        for batch_index in range(len(images)):
            pair = loader.dataset.pairs[offset + batch_index]
            truth = labels[batch_index]
            prediction = predictions[batch_index]
            loss_sum = F.cross_entropy(
                logits[batch_index:batch_index + 1],
                truth[None], reduction="sum"
            ).item()
            counts = {
                "loss": loss_sum,
                "pixels": truth.numel(),
                "tp": ((prediction == 1) & (truth == 1)).sum().item(),
                "fp": ((prediction == 1) & (truth == 0)).sum().item(),
                "fn": ((prediction == 0) & (truth == 1)).sum().item(),
                "tn": ((prediction == 0) & (truth == 0)).sum().item(),
                "samples": 1,
            }
            for key in totals:
                totals[key] += counts[key]
            source = pair.get("source", "default")
            aggregate = source_totals.setdefault(
                source, {key: 0 for key in totals}
            )
            for key in aggregate:
                aggregate[key] += counts[key]
            metrics = _metrics_from_counts(
                counts["loss"], counts["pixels"], counts["tp"], counts["fp"],
                counts["fn"], counts["tn"], 1
            )
            per_image.append({
                "image": pair["scan"],
                "source": source,
                **metrics,
            })
        offset += len(images)
    if not totals["pixels"]:
        raise ValueError("Cannot evaluate an empty dataset")
    aggregate = _metrics_from_counts(
        totals["loss"], totals["pixels"], totals["tp"], totals["fp"],
        totals["fn"], totals["tn"], totals["samples"]
    )
    aggregate["mean_image_dice"] = sum(row["dice"] for row in per_image) / len(per_image)
    aggregate["per_source"] = {
        source: _metrics_from_counts(
            values["loss"], values["pixels"], values["tp"], values["fp"],
            values["fn"], values["tn"], values["samples"]
        )
        for source, values in sorted(source_totals.items())
    }
    aggregate["per_image"] = per_image
    return aggregate

In [ ]:
"""Create the loss plot and portfolio-ready experiment report."""

import json
import csv
from pathlib import Path


def save_loss_plot(history, path, test_loss=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    epochs = [row["epoch"] for row in history]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, [row["train_loss"] for row in history], marker="o", label="Train")
    ax.plot(epochs, [row["loss"] for row in history], marker="o", label="Validation")
    if test_loss is not None:
        ax.axhline(test_loss, color="tab:green", linestyle="--",
                   label="Held out test (evaluated once)")
    ax.set(xlabel="Epoch", ylabel="Cross entropy loss", title="Segmentation loss")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def write_report(summary, metrics, preview_names, output_dir):
    output_dir = Path(output_dir)
    if not summary or not metrics:
        raise ValueError("Training summary and test metrics are required for a report")
    previews = "\n".join(
        f"![Scan, ground truth, prediction for {Path(name).stem}]({name})"
        for name in preview_names
    ) or "No previews were requested."
    architecture_note = (
        "No architecture deviation was used in this run."
        if summary["base_channels"] == 64
        else f"The first block used {summary['base_channels']} channels instead of the assignment's 64."
    )
    best_validation_dice = summary.get("best_validation_dice")
    best_validation_dice_row = (
        f"| Best validation Dice | {best_validation_dice:.4f} |\n"
        if best_validation_dice is not None else ""
    )
    source_rows = ""
    if metrics.get("per_source"):
        rows = ["| Source | Images | Dice | IoU | Precision | Recall |",
                "| --- | ---: | ---: | ---: | ---: | ---: |"]
        for source, values in metrics["per_source"].items():
            rows.append(
                f"| {source} | {values['samples']} | {values['dice']:.4f} | "
                f"{values['iou']:.4f} | {values['precision']:.4f} | {values['recall']:.4f} |"
            )
        source_rows = "\n## Performance by BBBC019 source\n\n" + "\n".join(rows) + "\n"
    dataset_note = ""
    if summary.get("dataset") == "BBBC019v2":
        used_pairs = sum(
            sum(summary.get("source_counts", {}).get(name, {}).values())
            for name in ("train", "val", "test")
        )
        pair_note = (
            f" This run used {used_pairs} verified image-mask pairs."
            if used_pairs else ""
        )
        dataset_note = (
            "\n## Dataset attribution\n\n"
            "This experiment uses [BBBC019v2](https://bbbc.broadinstitute.org/BBBC019) "
            "from the Broad Bioimage Benchmark Collection. The official page catalogs 171 "
            "manually segmented DIC microscopy images across eight source datasets."
            f"{pair_note} The dataset is licensed "
            "under CC BY 3.0. The dataset files are not redistributed in this repository.\n"
        )
    split_note = ""
    if summary.get("split_strategy") == "source_disjoint":
        split_note = (
            "\nThe split is source-disjoint: training uses "
            f"{', '.join(summary['train_sources'])}; validation uses "
            f"{', '.join(summary['val_sources'])}; and the held-out test uses "
            f"{', '.join(summary['test_sources'])}. This measures transfer to acquisition "
            "sources not seen during training.\n"
        )
    report = f"""# Cell segmentation results

This report was generated from a completed run of the assignment-style U-Net. The split was fixed before training. Validation selected the checkpoint; the held out test set was evaluated once.
{split_note}

## Configuration

| Setting | Value |
| --- | ---: |
| Dataset | {summary.get('dataset', 'custom paired dataset')} |
| Input image size | {summary['image_size']} x {summary['image_size']} |
| Output mask size | {summary['output_height']} x {summary['output_width']} |
| Train / validation / test images | {summary['train_samples']} / {summary['val_samples']} / {summary['test_samples']} |
| Epochs | {summary['epochs']} |
| Best validation epoch | {summary['best_epoch']} |
{best_validation_dice_row}| Batch size | {summary['batch_size']} |
| Learning rate | {summary['learning_rate']} |
| Device | {summary['device']} |
| Training time | {summary['training_seconds']:.1f} seconds |

The network uses unpadded 3 x 3 convolutions, transpose-convolution upsampling, and center-cropped skip connections. {architecture_note} Images are grayscale and normalized to [0, 1]. Training augmentation uses horizontal and vertical flips, 90-degree rotations, zooming, and gamma correction.

## Loss and test metrics

The chart shows training and validation loss at every epoch. The horizontal test line is the single held out test evaluation, not a per-epoch test measurement.

![Training, validation, and held out test loss](loss_curves.png)

| Held out test metric | Value |
| --- | ---: |
| Cross entropy loss | {metrics['loss']:.4f} |
| Pixel accuracy | {metrics['pixel_accuracy']:.4f} |
| Dice | {metrics['dice']:.4f} |
| Intersection over union | {metrics['iou']:.4f} |
| Precision | {metrics['precision']:.4f} |
| Recall | {metrics['recall']:.4f} |
| Mean per-image Dice | {metrics.get('mean_image_dice', metrics['dice']):.4f} |
| Test images | {metrics['samples']} |

{source_rows}
{dataset_note}

## Test segmentations

Each preview shows the centered scan crop, ground-truth mask, and prediction from left to right.

{previews}

The complete epoch history and raw metrics are saved in `history.json` and `metrics.json`. The exact split is also saved in `split.json` when that file is present in the exported results.
"""
    (output_dir / "README.md").write_text(report, encoding="utf-8")


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")


def write_per_image_csv(rows, path):
    fields = ["image", "source", "loss", "pixel_accuracy", "dice", "iou",
              "precision", "recall", "samples"]
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
with torch.no_grad():
    shape_model = UNet(base_channels=2).eval()
    shape_output = shape_model(torch.zeros(1, 1, 320, 320))
assert tuple(shape_output.shape) == (1, 2, 132, 132)
print('Shape check passed. Full model parameters:', sum(p.numel() for p in UNet(64).parameters()))

## 5. Train

Validation Dice selects the checkpoint. A scheduler halves the learning rate when validation loss stalls, and early stopping prevents unnecessary training. The held-out test set is not used here.

In [ ]:
import random
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

EPOCHS = 40
EARLY_STOPPING_PATIENCE = 8
BATCH_SIZE = 3
IMAGE_SIZE = 572
BASE_CHANNELS = 64
LEARNING_RATE = 1e-3
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

train_set = Cell_data(DATA_DIR, split['train'], IMAGE_SIZE, augment=True)
val_set = Cell_data(DATA_DIR, split['val'], IMAGE_SIZE, augment=False)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, pin_memory=device.type == 'cuda')
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, pin_memory=device.type == 'cuda')
model = UNet(BASE_CHANNELS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RUN_DIR / 'best_model.pt'
history = []
best_dice = -1.0
best_epoch = 0
epochs_without_improvement = 0
output_shape = None
start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_samples = 0
    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(images)
        output_shape = logits.shape[-2:]
        labels = center_crop(labels, *output_shape)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * len(images)
        train_samples += len(images)

    validation = evaluate(model, val_loader, device)
    scheduler.step(validation['loss'])
    record = {
        'epoch': epoch,
        'train_loss': train_loss_sum / train_samples,
        'learning_rate': optimizer.param_groups[0]['lr'],
        **validation,
    }
    history.append(record)
    write_json(RUN_DIR / 'history.json', history)
    if validation['dice'] > best_dice:
        best_dice = validation['dice']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'model_state': model.state_dict(),
            'image_size': IMAGE_SIZE,
            'base_channels': BASE_CHANNELS,
            'split': split,
            'epoch': epoch,
        }, CHECKPOINT_PATH)
    else:
        epochs_without_improvement += 1
    print(f"Epoch {epoch}/{EPOCHS}: train {record['train_loss']:.4f}, "
          f"val {validation['loss']:.4f}, Dice {validation['dice']:.4f}, "
          f"IoU {validation['iou']:.4f}, lr {record['learning_rate']:.2e}")
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print('Early stopping')
        break

training_seconds = time.perf_counter() - start
summary = {
    'dataset': split['dataset'],
    'split_strategy': split['split_strategy'],
    'train_sources': split['train_sources'],
    'val_sources': split['val_sources'],
    'test_sources': split['test_sources'],
    'image_size': IMAGE_SIZE,
    'output_height': output_shape[0],
    'output_width': output_shape[1],
    'base_channels': BASE_CHANNELS,
    'epochs': len(history),
    'epochs_requested': EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'seed': SEED,
    'train_samples': len(train_set),
    'val_samples': len(val_set),
    'test_samples': len(split['test']),
    'device': str(device),
    'training_seconds': training_seconds,
    'best_epoch': best_epoch,
    'best_validation_dice': best_dice,
    'source_counts': split['source_counts'],
}
write_json(RUN_DIR / 'training_summary.json', summary)
save_loss_plot(history, RUN_DIR / 'loss_curves.png')
print(f'Best epoch {best_epoch}, validation Dice {best_dice:.4f}, time {training_seconds:.1f}s')

## 6. Held-out evaluation

The selected checkpoint is evaluated once on 26 held-out images. Metrics are calculated across all pixels, for each image, and separately for the two held-out test sources.

In [ ]:
import shutil

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
verified_split = load_split(SPLIT_PATH, DATA_DIR)
if checkpoint['split'] != verified_split:
    raise ValueError('Split does not match the trained checkpoint')

test_set = Cell_data(DATA_DIR, verified_split['test'], checkpoint['image_size'], augment=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, pin_memory=device.type == 'cuda')
test_model = UNet(checkpoint['base_channels']).to(device)
test_model.load_state_dict(checkpoint['model_state'])
test_metrics = evaluate_detailed(test_model, test_loader, device)
test_metrics['checkpoint_epoch'] = checkpoint['epoch']
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
write_json(RESULTS_DIR / 'metrics.json', test_metrics)
write_per_image_csv(test_metrics['per_image'], RESULTS_DIR / 'per_image_metrics.csv')
print('Aggregate test metrics:')
for key in ('loss', 'pixel_accuracy', 'dice', 'iou', 'precision', 'recall', 'mean_image_dice'):
    print(f'{key}: {test_metrics[key]:.4f}')
print()
print('Per-source Dice:')
for source, values in test_metrics['per_source'].items():
    print(f"{source:15s} {values['dice']:.4f} ({values['samples']} images)")

## 7. Export predictions and portfolio report

Every test preview shows the centered scan crop, manual foreground mask, and prediction from left to right. The report includes configuration, aggregate metrics, per-source metrics, loss curves, attribution, and all previews.

In [ ]:
preview_names = []
test_model.eval()
with torch.no_grad():
    for index in range(len(test_set)):
        image, label = test_set[index]
        prediction = test_model(torch.from_numpy(image[None]).to(device)).argmax(dim=1)[0].cpu().numpy()
        height, width = prediction.shape
        image_crop = center_crop(image[0], height, width)
        label_crop = center_crop(label, height, width)
        panels = [
            Image.fromarray((image_crop * 255).astype(np.uint8)),
            Image.fromarray((label_crop * 255).astype(np.uint8)),
            Image.fromarray((prediction * 255).astype(np.uint8)),
        ]
        preview = Image.new('L', (width * 3, height))
        for column, panel in enumerate(panels):
            preview.paste(panel, (column * width, 0))
        pair = test_set.pairs[index]
        name = f"{pair['source']}_{Path(pair['scan']).stem}_preview.png"
        preview.save(RESULTS_DIR / name)
        preview_names.append(name)

save_loss_plot(history, RESULTS_DIR / 'loss_curves.png', test_metrics['loss'])
shutil.copyfile(SPLIT_PATH, RESULTS_DIR / 'split.json')
shutil.copyfile(RUN_DIR / 'history.json', RESULTS_DIR / 'history.json')
write_json(RESULTS_DIR / 'training_summary.json', summary)
write_report(summary, test_metrics, preview_names, RESULTS_DIR)
archive_path = shutil.make_archive('/content/bbbc019_portfolio_results', 'zip', root_dir='/content/results', base_dir='bbbc019')
print('Report:', RESULTS_DIR / 'README.md')
print('Archive:', archive_path)

## 8. Inspect and download

Read the generated report and inspect representative predictions before publishing. The zip excludes the dataset and model checkpoint.

In [ ]:
from IPython.display import display

print((RESULTS_DIR / 'README.md').read_text(encoding='utf-8'))
display(Image.open(RESULTS_DIR / 'loss_curves.png'))
shown_sources = set()
for pair, name in zip(test_set.pairs, preview_names):
    if pair['source'] not in shown_sources:
        shown_sources.add(pair['source'])
        print(pair['source'], ': scan | ground truth | prediction')
        display(Image.open(RESULTS_DIR / name))
try:
    from google.colab import files
    files.download('/content/bbbc019_portfolio_results.zip')
except ImportError:
    print('Download /content/bbbc019_portfolio_results.zip manually.')